In [1]:
import os
import numpy as np
import pandas as pd
import joblib

# Random Forest Models
from xgboost import XGBRegressor
# Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Model Tuning
import optuna

/home/aliarridha/miniconda3/envs/modeling1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setup path
BASE_PATH = 'dataset'
OUTPUT_FOLDER = os.path.join(BASE_PATH, 'output_pipeline')
ARTIFACTS_FOLDER = os.path.join(BASE_PATH, 'artifacts')
MODEL_FOLDER = os.path.join(BASE_PATH, 'model_outputs')
EDA_FOLDER = os.path.join(BASE_PATH, 'eda_outputs')
RESULTS_FOLDER = os.path.join(BASE_PATH, 'results')

os.makedirs(MODEL_FOLDER, exist_ok=True)
os.makedirs(EDA_FOLDER, exist_ok=True)
os.makedirs(RESULTS_FOLDER, exist_ok=True)

In [3]:
# Load fitur (X)
X_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_train.parquet'))
X_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_val.parquet'))
X_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_test.parquet'))

In [4]:
# Load target (y) - convert ke Series untuk sklearn
y_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_train.parquet'))
y_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_val.parquet'))
y_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_test.parquet'))

In [5]:
# Load metadata ID (untuk traceback hasil prediksi nanti)
ID_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_train.parquet'))
ID_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_val.parquet'))
ID_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_test.parquet'))

In [6]:
scaler = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'minmax_scaler.joblib'))
winsor = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'winsor_bounds.joblib'))
feat_select = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'feature_selection_info.joblib'))

/home/aliarridha/miniconda3/envs/modeling1/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
TARGET = feat_select['target']

In [8]:
y_train_arr = y_train[TARGET].values
y_val_arr = y_val[TARGET].values
y_test_arr = y_test[TARGET].values

In [9]:
print(ID_test.head())
print(ID_test.columns)

         kabupaten_kota   nama_wilayah_bersih  tahun
0       Kab. Aceh Barat       kab. aceh barat   2024
1       Kab. Aceh Barat       kab. aceh barat   2025
2  Kab. Aceh Barat Daya  kab. aceh barat daya   2024
3  Kab. Aceh Barat Daya  kab. aceh barat daya   2025
4       Kab. Aceh Besar       kab. aceh besar   2024
Index(['kabupaten_kota', 'nama_wilayah_bersih', 'tahun'], dtype='object')


In [10]:
print("Mulai tuning XGBoost (Bayesian - Optuna)")

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.1),
        "max_depth": trial.suggest_int("max_depth", 4, 8),
        "subsample": trial.suggest_float("subsample", 0.7, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 0.9),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
    }

    model = XGBRegressor(
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror",
        eval_metric="mae",
        **params,
    )
    model.fit(X_train, y_train_arr)
    val_pred = model.predict(X_val)
    val_mae = mean_absolute_error(y_val_arr, val_pred)
    return val_mae

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

best_params = study.best_params
best_mae = study.best_value

xgb_model = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror",
    eval_metric="mae",
    **best_params,
)
xgb_model.fit(X_train, y_train_arr)

print("Tuning selesai")
print("Best params:")
print(best_params)
print(f"Best Validation MAE: {best_mae:.4f}")

[I 2026-06-04 16:00:31,541] A new study created in memory with name: no-name-f8f89cc1-0652-4eeb-878a-6092a5646ef5


Mulai tuning XGBoost (Bayesian - Optuna)


Best trial: 0. Best value: 3.95615:   3%|▎         | 1/30 [00:01<00:45,  1.57s/it]

[I 2026-06-04 16:00:33,112] Trial 0 finished with value: 3.9561542175292965 and parameters: {'n_estimators': 350, 'learning_rate': 0.06313496286315998, 'max_depth': 7, 'subsample': 0.8756425627687613, 'colsample_bytree': 0.7322912830355056, 'min_child_weight': 4}. Best is trial 0 with value: 3.9561542175292965.


Best trial: 1. Best value: 3.93663:   7%|▋         | 2/30 [00:02<00:36,  1.31s/it]

...

[I 2026-06-04 16:01:07,671] Trial 27 finished with value: 4.0412779990302194 and parameters: {'n_estimators': 400, 'learning_rate': 0.04357985106147711, 'max_depth': 7, 'subsample': 0.7950400677528675, 'colsample_bytree': 0.7688251827452964, 'min_child_weight': 3}. Best is trial 1 with value: 3.9366338575575086.


Best trial: 28. Best value: 3.89566:  97%|█████████▋| 29/30 [00:38<00:01,  1.46s/it]

[I 2026-06-04 16:01:09,653] Trial 28 finished with value: 3.8956585456000434 and parameters: {'n_estimators': 450, 'learning_rate': 0.058146570402758335, 'max_depth': 8, 'subsample': 0.8823112488822615, 'colsample_bytree': 0.8123044775099262, 'min_child_weight': 1}. Best is trial 28 with value: 3.8956585456000434.


Best trial: 28. Best value: 3.89566: 100%|██████████| 30/30 [00:39<00:00,  1.30s/it]


[I 2026-06-04 16:01:10,615] Trial 29 finished with value: 4.010991648017035 and parameters: {'n_estimators': 350, 'learning_rate': 0.06443982513581428, 'max_depth': 7, 'subsample': 0.886280560729129, 'colsample_bytree': 0.811248501954843, 'min_child_weight': 1}. Best is trial 28 with value: 3.8956585456000434.
Tuning selesai
Best params:
{'n_estimators': 450, 'learning_rate': 0.058146570402758335, 'max_depth': 8, 'subsample': 0.8823112488822615, 'colsample_bytree': 0.8123044775099262, 'min_child_weight': 1}
Best Validation MAE: 3.8957


In [11]:
# Prediksi
xgb_train_pred = xgb_model.predict(X_train)
xgb_val_pred = xgb_model.predict(X_val)
xgb_test_pred = xgb_model.predict(X_test)

In [12]:
def hitung_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

In [13]:
# evaluasi train
xgb_train_mae, xgb_train_rmse, xgb_train_r2 = hitung_metrics(
    y_train,
    xgb_train_pred
)

In [14]:
# evaluasi validation
xgb_val_mae, xgb_val_rmse, xgb_val_r2 = hitung_metrics(
    y_val,
    xgb_val_pred
)

In [15]:
# evaluasi test
xgb_test_mae, xgb_test_rmse, xgb_test_r2 = hitung_metrics(
    y_test,
    xgb_test_pred
)

In [16]:
print("HASIL XGBoost")

print("\nTrain")
print(f"MAE  : {xgb_train_mae:.4f}")
print(f"RMSE : {xgb_train_rmse:.4f}")
print(f"R2   : {xgb_train_r2:.4f}")

print("\nValidation")
print(f"MAE  : {xgb_val_mae:.4f}")
print(f"RMSE : {xgb_val_rmse:.4f}")
print(f"R2   : {xgb_val_r2:.4f}")

print("\nTest")
print(f"MAE  : {xgb_test_mae:.4f}")
print(f"RMSE : {xgb_test_rmse:.4f}")
print(f"R2   : {xgb_test_r2:.4f}")

HASIL XGBoost

Train
MAE  : 0.3918
RMSE : 0.6102
R2   : 0.9968

Validation
MAE  : 3.8957
RMSE : 5.0609
R2   : 0.7581

Test
MAE  : 4.2120
RMSE : 5.5236
R2   : 0.7263


In [17]:
# Simpan Model XGBoost
joblib.dump(
    xgb_model,
    os.path.join(BASE_PATH, 'models', 'xgboost.joblib')
)
print("Model berhasil disimpan.")

Model berhasil disimpan.


In [18]:
# Simpan metrics
metrics_xgb = pd.DataFrame({
    "model": ["XGBoost"],
    "train_mae": [xgb_train_mae],
    "train_rmse": [xgb_train_rmse],
    "train_r2": [xgb_train_r2],
    "val_mae": [xgb_val_mae],
    "val_rmse": [xgb_val_rmse],
    "val_r2": [xgb_val_r2],
    "test_mae": [xgb_test_mae],
    "test_rmse": [xgb_test_rmse],
    "test_r2": [xgb_test_r2]
})



os.makedirs("results", exist_ok=True)

metrics_xgb.to_csv(
    os.path.join(BASE_PATH, "results", "xgboost_metrics.csv"),
    index=False
)
print("Metrics berhasil disimpan.")

Metrics berhasil disimpan.


In [19]:
# Simpan hasil prediksi test
hasil_pred_lr = ID_test.copy()

hasil_pred_lr["actual"] = y_test.values
hasil_pred_lr["prediction_xgb"] = xgb_test_pred

hasil_pred_lr.to_csv(
    os.path.join(BASE_PATH, "results", "xgboost_predictions.csv"),
    index=False
)

print("Hasil prediksi berhasil disimpan.")

# preview hasil
print("\nPreview prediction:")
print(hasil_pred_lr.head())

Hasil prediksi berhasil disimpan.

Preview prediction:
         kabupaten_kota   nama_wilayah_bersih  tahun  actual  prediction_xgb
0       Kab. Aceh Barat       kab. aceh barat   2024   54.82       46.722130
1       Kab. Aceh Barat       kab. aceh barat   2025   53.53       51.363571
2  Kab. Aceh Barat Daya  kab. aceh barat daya   2024   55.15       54.137527
3  Kab. Aceh Barat Daya  kab. aceh barat daya   2025   66.77       55.376797
4       Kab. Aceh Besar       kab. aceh besar   2024   51.82       46.552723


In [20]:
feature_importance_xgb = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_model.feature_importances_
})

feature_importance_xgb = feature_importance_xgb.sort_values(
    by="importance",
    ascending=False
)

feature_importance_xgb.to_csv(
    os.path.join(BASE_PATH, "results", 'xgboost_feature_importance.csv'),
    index=False
)

print("Hasil XGBoost feature importance berhasil disimpan.")

Hasil XGBoost feature importance berhasil disimpan.


In [21]:
joblib.dump(
    xgb_model,
    os.path.join(
        MODEL_FOLDER,
        'xgboost_model.joblib'
    )
)

print("XGBoost model berhasil disimpan.")

XGBoost model berhasil disimpan.
